In [ ]:
!pip install bitsandbytes

In [ ]:
!pip install --upgrade transformers accelerate bitsandbytes

In [ ]:
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

**FP32 (Full Precision, 4 Bytes):** In FP32, 8 bits are reserved for the "exponent", 23 bits for the "mantissa" and 1 bit for the sign of the number.
Se prueba a entrenar con half precission (fp16) por precisión parecida y la mitad de tamaño.


In [3]:
import os, sys, copy, time, math
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from vvtk_quant import VVTK_QModel
from vvtk_quant.quant import nearest_index
from common_seq import DEVICE, OUT, tiny_shakespeare, CharLM, lm_batch, lm_eval, lm_sample, cosine
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
torch.backends.cuda.matmul.allow_tf32 = True

import bitsandbytes as bnb
from bitsandbytes.nn import Linear8bitLt
import copy
from accelerate.utils import BnbQuantizationConfig, load_and_quantize_model
from accelerate import init_empty_weights

**1. Cuantización de un modelo completo a 8 bits**

In [5]:
train_ids, val_ids, itos = tiny_shakespeare()
print(f"train {len(train_ids):,} chars, val {len(val_ids):,} chars, vocabulary {len(itos)}")
CTX, BATCH = 256, 64
torch.manual_seed(0)
model = CharLM(len(itos), d=256, layers=4, heads=4).to(DEVICE)
n_lin = sum(m.weight.numel() for m in model.modules() if isinstance(m, nn.Linear))
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M, of which in nn.Linear: {n_lin / 1e6:.2f} M")

train 1,003,854 chars, val 111,540 chars, vocabulary 65
parameters: 3.19 M, of which in nn.Linear: 3.16 M


In [6]:
def train_lm(model, steps, lr, step_hook=None, seed=0, log_every=500, wd=0.1):
    opt = torch.optim.AdamW(model.parameters(), lr, betas=(0.9, 0.95), weight_decay=wd)
    sched = cosine(opt, steps, warmup=min(200, steps // 10))
    g = torch.Generator().manual_seed(seed)
    curve = []
    t0 = time.time()
    for s in range(steps):
        model.train()
        if step_hook is not None:
            step_hook(s, steps)
        x, y = lm_batch(train_ids, BATCH, CTX, g)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
            loss = F.cross_entropy(model(x).flatten(0, 1).float(), y.flatten())
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        if (s + 1) % log_every == 0 or s + 1 == steps:
            v = lm_eval(model, val_ids, CTX)
            curve.append((s + 1, v))
            print(f"    step {s + 1:>5}/{steps}  train {loss.item():.3f}  val {v:.3f}  ({time.time() - t0:.0f}s)")
    return curve

ckpt = os.path.join(OUT, "charlm_fp32.pth")
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)); print("loaded", ckpt)
else:
    train_lm(model, 3000, 1e-3)
    torch.save(model.state_dict(), ckpt)
val_fp32 = lm_eval(model, val_ids, CTX)

loaded /extra/scratch05/alumno1/vvtk_quant/examples/out/charlm_fp32.pth


In [7]:
# guardo el modelo entrenado de la tinyLLM
torch.save(model.state_dict(), "model.pt")

In [8]:
def replace_8bit_linear(model, threshold=6.0, module_to_not_convert="lm_head"):
    for name, module in model.named_children():
        if len(list(module.children())) > 0:
            replace_8bit_linear(module, threshold, module_to_not_convert)

        if isinstance(module, nn.Linear) and name != module_to_not_convert:
            with init_empty_weights():
                model._modules[name] = bnb.nn.Linear8bitLt(
                    module.in_features,
                    module.out_features,
                    module.bias is not None,
                    has_fp16_weights=False,
                    threshold=threshold,
                )
    return model

In [9]:
# Copia del modelo entrenado, convertida a FP16
model_fp16 = copy.deepcopy(model).to(
    device="cpu",
    dtype=torch.float16
)

# Configuración INT8
bnb_config = BnbQuantizationConfig(
    load_in_8bit=True,
    torch_dtype=torch.float16,
    llm_int8_threshold=6.0,
    skip_modules=[]
)

# Cuantización
model_int8 = load_and_quantize_model(
    model_fp16,
    bnb_quantization_config=bnb_config
)

model_int8.eval()
val_int8 = lm_eval(model_int8, val_ids, CTX)

It is not recommended to quantize a loaded model. The model should be instantiated under the `init_empty_weights` context manager.


**Comparar la inferencia (texto generado) con cada modelo**

In [7]:
# Inferencia con el modelo original
print("FP32:")
print(lm_sample(model, itos, n=200))

# Inferencia con el modelo cuantizado
print("INT8:")
print(lm_sample(model_int8, itos, n=200))

FP32:
ROMEO:
I am I not like mointy should be a thing.

BUSHY:
Why, fellow, sir?

BALTHASAR:
I shall be not a little o'er deserves you.

KING HENRY VI:
Be taken so sin, what deadly day is dead
That by the instruc
INT8:
ROMEO:
I am I not like mointy should be a thing.

BUCKINGHAM:
That's not confess, by the skilful lords.

GLOUCESTER:
Now, noble partner than my sister hath been
The arm enough and towards Marcius benefit
In


**Número de parámetros cuantificados**

In [10]:
linear_int8 = sum(
    isinstance(m, bnb.nn.Linear8bitLt)
    for m in model_int8.modules()
)
linear_FP32 = sum(
    isinstance(m, nn.Linear)
    for m in model.modules()
)


print("Capas INT8:", linear_int8)
print("Con FP32 teníamos",linear_FP32)

Capas INT8: 25
Con FP32 teníamos 25


**Memoria ocupada por los pesos**

In [11]:
mem_fp32 = sum(
    t.numel() * t.element_size()
    for t in model.state_dict().values()
)

mem_int8 = sum(
    t.numel() * t.element_size()
    for t in model_int8.state_dict().values()
)

print(f"FP32: {mem_fp32 / 1024**2:.2f} MB")
print(f"INT8: {mem_int8 / 1024**2:.2f} MB")

print(
    f"Reducción: {(1 - mem_int8 / mem_fp32) * 100:.2f} %"
)

FP32: 12.16 MB
INT8: 3.10 MB
Reducción: 74.50 %


**Val loss**

In [12]:
print(f"FP32: {val_fp32:.4f}")
print(f"INT8: {val_int8:.4f}")
print(f"Empeora: {np.double((val_int8 - val_fp32)/val_fp32*100):.4f} %") # menor=mejor

FP32: 1.4937
INT8: 1.4939
Empeora: 0.0163 %


**2. Cuantización a 4 bits con NF4**

In [13]:
from bitsandbytes.nn import Linear4bit
from accelerate.utils import BnbQuantizationConfig, load_and_quantize_model
# Configuración INT4
quantization_config=BnbQuantizationConfig(load_in_4bit=True,torch_dtype=torch.float16,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=False,skip_modules=[])
# Cuantización
model_fp16_4bit=copy.deepcopy(model).to(device="cpu",dtype=torch.float16)
model_4bit = load_and_quantize_model(model_fp16_4bit,bnb_quantization_config=quantization_config)
model_4bit.eval()
val_4bit = lm_eval(model_4bit, val_ids, CTX)

It is not recommended to quantize a loaded model. The model should be instantiated under the `init_empty_weights` context manager.


**Comparar la inferencia (texto generado) con cada modelo**

In [12]:
# Inferencia con el modelo original
print("FP32:")
print(lm_sample(model, itos, n=200))

# Inferencia con el modelo cuantizado
print("INT4bit:")
print(lm_sample(model_4bit, itos, n=200))

FP32:
ROMEO:
I am I not like mointy should be a thing.

BUSHY:
Why, fellow, sir?

BALTHASAR:
I shall be not a little o'er deserves you.

KING HENRY VI:
Be taken so sin, what deadly day is dead
That by the instruc
INT4bit:
ROMEO:
I will not like moiety should be to him.
Take her not give my heart to repent:
I do it like good lord. You then I am such
to hardly remember a man beggar terrand
Nothing but the market-black of beaut


**Número de parámetros cuantificados**

In [22]:
linear_4bit = sum(
    isinstance(m, bnb.nn.Linear4bit)
    for m in model_4bit.modules()
)
linear_FP32 = sum(
    isinstance(m, nn.Linear)
    for m in model.modules()
)


print("Capas INT4bit:", linear_4bit)
print("Con FP32 teníamos",linear_FP32)

Capas INT4bit: 25
Con FP32 teníamos 25


**Memoria ocupada por los pesos**

In [14]:
mem_int4 = sum(
    t.numel() * t.element_size()
    for t in model_4bit.state_dict().values()
)

print(f"FP32: {mem_fp32 / 1024**2:.2f} MB")
print(f"INT4bit: {mem_int4 / 1024**2:.2f} MB")

print(
    f"Reducción: {(1 - mem_int4 / mem_fp32) * 100:.2f} %"
)

FP32: 12.16 MB
INT4bit: 1.75 MB
Reducción: 85.61 %


**Val loss**

In [15]:
print(f"FP32: {val_fp32:.4f}")
print(f"INT4bit: {val_4bit:.4f}")
print(f"Empeora: {np.double((val_4bit - val_fp32)/val_fp32*100):.4f} %") # menor=mejor

FP32: 1.4937
INT4bit: 1.4983
Empeora: 0.3105 %


**Comparación con la tinyLLM con 2 bits soft-lloyd (no anchor) teniendo QAT**

In [16]:
def wrap(m, b, method, **kw):
    q = VVTK_QModel(copy.deepcopy(m), b=b, module_types=[nn.Linear], **kw)
    q.calibrate_quant(method=method)
    return q

In [17]:
QSTEPS, QLR, T0, T1 = 600, 3e-4, 2.0, 40.0
CONTENDERS = {
    "fp32":              None,
    "soft-lloyd":        dict(mode="soft", method="lloyd"),
}

def qat(name, b):
    cfg = CONTENDERS[name]
    torch.manual_seed(0)
    if cfg is None:
        q = copy.deepcopy(model)
    else:
        cfg = dict(cfg); meth = cfg.pop("method")
        q = wrap(model, b, meth, temp=T0, **cfg)
    def hook(s, n):
        if cfg is not None and cfg["mode"] == "soft":
            q.update_temp(T0 * (T1 / T0) ** (s / (n - 1)))
    start = lm_eval(q, val_ids, CTX)
    curve = [(0, start)] + train_lm(q, QSTEPS, QLR, step_hook=hook, log_every=100, wd=0.0)
    return q, curve

R = {}

# Guardamos sus resultados
R[("FP32", 0)] = (model, [(0, val_fp32)])
R[("INT8", 0)] = (model_int8, [(0, val_int8)])
R[("INT4", 0)] = (model_4bit, [(0, val_4bit)])

# Experimentos VVTK con QAT
for b in [3, 2]:
    print(f"--- soft-lloyd, {b} bits")
    R[("soft-lloyd", b)] = qat("soft-lloyd", b)

--- soft-lloyd, 3 bits
    step   100/600  train 1.171  val 1.519  (35s)
    step   200/600  train 1.148  val 1.521  (69s)
    step   300/600  train 1.138  val 1.508  (103s)
    step   400/600  train 1.164  val 1.506  (138s)
    step   500/600  train 1.130  val 1.500  (172s)
    step   600/600  train 1.121  val 1.498  (206s)
--- soft-lloyd, 2 bits
    step   100/600  train 1.214  val 1.646  (34s)
    step   200/600  train 1.177  val 1.613  (69s)
    step   300/600  train 1.166  val 1.559  (103s)
    step   400/600  train 1.199  val 1.529  (138s)
    step   500/600  train 1.169  val 1.516  (172s)
    step   600/600  train 1.169  val 1.513  (207s)


**4 bits usando fp4 (en lugar de nf4)**

In [18]:
# Configuración INT4
quantization_config=BnbQuantizationConfig(load_in_4bit=True,torch_dtype=torch.float16,bnb_4bit_quant_type="fp4",bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=False,skip_modules=[])
# Cuantización
model_fp16_fp4=copy.deepcopy(model).to(device="cpu",dtype=torch.float16)
model_fp4 = load_and_quantize_model(model_fp16_fp4,bnb_quantization_config=quantization_config)
model_fp4.eval()
val_fp4 = lm_eval(model_fp4, val_ids, CTX)

It is not recommended to quantize a loaded model. The model should be instantiated under the `init_empty_weights` context manager.


**Memoria ocupada por los pesos**

In [19]:
mem_fp4 = sum(
    t.numel() * t.element_size()
    for t in model_fp4.state_dict().values()
)

print(f"FP32: {mem_fp32 / 1024**2:.2f} MB")
print(f"INT4bit FP4: {mem_fp4 / 1024**2:.2f} MB")

print(
    f"Reducción: {(1 - mem_fp4 / mem_fp32) * 100:.2f} %"
)

FP32: 12.16 MB
INT4bit FP4: 1.75 MB
Reducción: 85.61 %


In [20]:
# calculo memoria de los modelos
"""
mem_softlloyd_3b = sum(
    t.numel() * t.element_size()
    for t in R[("soft-lloyd", 3)][0].state_dict().values()
)
mem_softlloyd_2b = sum(
    t.numel() * t.element_size()
    for t in R[("soft-lloyd", 2)][0].state_dict().values()
)
"""
# Copias para no modificar los modelos originales de QAT
q3 = copy.deepcopy(R[("soft-lloyd", 3)][0])
q2 = copy.deepcopy(R[("soft-lloyd", 2)][0])

# Aplicamos la cuantización definitiva
q3.apply_quant()
q2.apply_quant()

# Guardamos los modelos comprimidos
q3.save("soft_lloyd_3bit.q.gz")
q2.save("soft_lloyd_2bit.q.gz")
# Guardar modelo INT8 y 4
torch.save(model_int8.state_dict(), "model_int8.pth")
torch.save(model_4bit.state_dict(), "model_4bit.pth")

# Calculamos el tamaño de los archivos
mem_softlloyd_3b = os.path.getsize("soft_lloyd_3bit.q.gz")
mem_softlloyd_2b = os.path.getsize("soft_lloyd_2bit.q.gz")
mem_int8 = os.path.getsize("model_int8.pth")
mem_int4 = os.path.getsize("model_4bit.pth")

memorias=[mem_fp32,mem_int8,mem_int4,mem_fp4,mem_softlloyd_3b,mem_softlloyd_2b]

names = [
    "FP32",
    "INT8",
    "INT4 NF4",
    "INT4 FP4",
    "Soft-Lloyd 3-bit",
    "Soft-Lloyd 2-bit"
]

losses = [
    val_fp32,
    val_int8,
    val_4bit,
    val_fp4,
    R[("soft-lloyd", 3)][1][-1][1],
    R[("soft-lloyd", 2)][1][-1][1]
]


No puedo contar la memoria del modelo de soft lloyd de la primera forma que hay comentada porque **state_dict()** contabiliza los bytes ocupados por los tensores que forman el estado del modelo, mientras que **os.path.getsize(...)** contabiliza los bytes del archivo guardado en disco, incluyendo la representación concreta que utiliza el método de guardado. El primer método tiene un problema, y es que en Soft-Lloyd el modelo puede conservar pesos originales en FP32 y estructuras auxiliares utilizadas durante el entrenamiento. Usamos la misma métrica para medir el tamaño de las representaciones de 4 y 8 bits, aunque ya hubiéramos calculado el tamaño que ocupan en memoria de la otra forma previamente.

In [21]:
print(f"{'Modelo':<22} | {'Validation loss':>16} | {'Memoria ocupada (MB)':>16}")
print("-" * 63)

for name, loss, mem in zip(names, losses, memorias):
    print(f"{name:<22} | {loss:16.4f} | {mem/1024**2:16.4f}")

Modelo                 |  Validation loss | Memoria ocupada (MB)
---------------------------------------------------------------
FP32                   |           1.4937 |          12.1641
INT8                   |           1.4939 |           3.1327
INT4 NF4               |           1.4983 |           1.7894
INT4 FP4               |           1.5040 |           1.7501
Soft-Lloyd 3-bit       |           1.4984 |           1.2002
Soft-Lloyd 2-bit       |           1.5126 |           0.8198


Soft-Lloyd de b=2 es la mejor alternativa, aunque tarde mucho más que los demás modelos: comprime mucho mejor. Merece la pena probar con 4bits nf4, al ser mucho más rápido y lograr también una compresión bastante buena.

**Cuantización usando QLoRA**

Añade adaptadores QLoRA a la cuantificación de 4 bits

In [36]:
!pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 19.3 MB/s  0:00:00


In [22]:
from peft import LoraConfig, get_peft_model
model_qlora = copy.deepcopy(model_4bit)
lora_config = LoraConfig(r=8,lora_alpha=16,target_modules="all-linear",lora_dropout=0.05,bias="none")
model_qlora = get_peft_model(model_qlora, lora_config)
loss_qlora_initial = lm_eval(model_qlora, val_ids, CTX)
print(f"QLoRA antes del entrenamiento: {loss_qlora_initial:.4f} (se supone que tiene que coincidir con la que teníamos para model_4bit)")

QLoRA antes del entrenamiento: 1.4983 (se supone que tiene que coincidir con la que teníamos para model_4bit)


In [23]:
# entrenamos los adaptadores LoRA con los pesos en int4 congelados
ckpt = os.path.join(OUT, "charlm_qlora")
if os.path.exists(ckpt):
    model_qlora.load_adapter(ckpt, adapter_name="default")
    print("Adaptadores cargados:", ckpt)
else:
    train_lm(model_qlora, 3000, 1e-3)
    model_qlora.save_pretrained(ckpt)
    print("Adaptadores guardados:", ckpt)
val_qlora = lm_eval(model_qlora, val_ids, CTX)
print(f"Loss QLoRA después del entrenamiento: {val_qlora:.4f}")

Adaptadores cargados: /extra/scratch05/alumno1/vvtk_quant/examples/out/charlm_qlora
Loss QLoRA después del entrenamiento: 1.4980


**Val loss**

In [24]:
print(f"FP32: {val_fp32:.4f}")
print(f"INT4bit con qlora: {val_qlora:.4f}")
print(f"Empeora: {np.double((val_qlora - val_fp32)/val_fp32*100):.4f} %") # menor=mejor

FP32: 1.4937
INT4bit con qlora: 1.4980
Empeora: 0.2909 %


**Inferencia**

In [25]:
# Inferencia con el modelo cuantizado
print("INT4bit con qlora:")
print(lm_sample(model_qlora, itos, n=200))

INT4bit con qlora:
ROMEO:
I am the seven to injuries, whence thou wilt not.

NORTHUMBERLAND:
Why, then he should live groan'd to him?

BUSHY:
Marry, my gracious lord.

FRIAR LAURENCE:
Bliss one Lucio, do me from the house.

N


**Memoria ocupada por los pesos**

In [26]:
mem_qlora1 = sum(
    t.numel() * t.element_size()
    for t in model_qlora.state_dict().values()
)

print(f"FP32: {mem_fp32 / 1024**2:.2f} MB")
print(f"INT4 NF4 con qlora: {mem_qlora1 / 1024**2:.2f} MB")

print(
    f"Reducción: {(1 - mem_qlora1 / mem_fp32) * 100:.2f} %"
)

FP32: 12.16 MB
INT4 NF4 con qlora: 2.32 MB
Reducción: 80.91 %


In [28]:
torch.save(model_qlora.state_dict(), "model_qlora.pth")
mem_qlora = os.path.getsize("model_qlora.pth")
memorias=[mem_fp32,mem_int8,mem_int4,mem_fp4,mem_softlloyd_3b,mem_softlloyd_2b,mem_qlora]

names = [
    "FP32",
    "INT8",
    "INT4 NF4",
    "INT4 FP4",
    "Soft-Lloyd 3-bit",
    "Soft-Lloyd 2-bit",
    "INT4 NF4 con QLoRA"
]

losses = [
    val_fp32,
    val_int8,
    val_4bit,
    val_fp4,
    R[("soft-lloyd", 3)][1][-1][1],
    R[("soft-lloyd", 2)][1][-1][1],
    val_qlora
]
print(f"{'Modelo':<22} | {'Validation loss':>16} | {'Memoria ocupada (MB)':>16}")
print("-" * 73)

for name, loss, mem in zip(names, losses, memorias):
    print(f"{name:<22} | {loss:16.4f} | {mem/1024**2:16.4f}")

Modelo                 |  Validation loss | Memoria ocupada (MB)
-------------------------------------------------------------------------
FP32                   |           1.4937 |          12.1641
INT8                   |           1.4939 |           3.1327
INT4 NF4               |           1.4983 |           1.7894
INT4 FP4               |           1.5040 |           1.7501
Soft-Lloyd 3-bit       |           1.4984 |           1.2002
Soft-Lloyd 2-bit       |           1.5126 |           0.8198
INT4 NF4 con QLoRA     |           1.4980 |           2.3980


NF4 consigue menor validation loss (mejor inferencia) que FP4, pero ocupa algo más de memoria (peor compresión)

In [29]:
# separamos la memoria base del modelo de la de los adaptadores qlora
def memory_breakdown(model):
    base_bytes = 0
    lora_bytes = 0

    for name, tensor in model.state_dict().items():
        size = tensor.numel() * tensor.element_size()

        if "lora_" in name:
            lora_bytes += size
        else:
            base_bytes += size

    print(f"Modelo base (INT4 NF4): {base_bytes / 1024**2:.4f} MB")
    print(f"Adaptadores LoRA: {lora_bytes / 1024**2:.4f} MB")
    print(f"Total: {(base_bytes + lora_bytes) / 1024**2:.4f} MB")

memory_breakdown(model_qlora)

Modelo base (INT4 NF4): 1.7501 MB
Adaptadores LoRA: 0.5723 MB
Total: 2.3224 MB
